# BERT và GPT: cùng một khối Transformer, khác mask và mục tiêu train

BERT (encoder) và GPT (decoder) dùng **cùng** khối Transformer. Chỉ hai thứ khác nhau: mask attention (hai chiều hay nhân
quả) và mục tiêu train (đoán token bị che, masked language modeling, hay đoán token kế tiếp). Bạn dựng một lớp
`TinyTransformer` có cờ `causal`, train hai bản trên cùng 1 000 bài Wikipedia tiếng Việt với cùng số bước, rồi đo hai việc
mà mỗi bên giỏi: điền chỗ trống giữa câu, và sinh văn bản.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, lớp. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, train, vẽ; test bỏ qua chúng.
- Mặc định chạy trên CPU (khoảng 10 phút cả notebook); kernel `ai-gpu` tự chọn cấu hình lớn hơn.
- Chấm: `uv run pytest tracks/ai-engineer/modules/05-transformers-llm-architecture/exercises/intermediate/02-bert-vs-gpt -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, kiểm Wikipedia tiếng Việt
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
WIKI = ROOT / "data/raw/wikipedia-vi"
if not (WIKI.is_dir() and any(WIKI.iterdir())):
    sys.path.insert(0, str(ROOT))
    from data.download_dataset import ensure

    ensure("wikipedia-vi")
SHARD = WIKI / "20231101.vi/train-00002-of-00004.parquet"

In [ ]:
import math
import time
import unicodedata

import numpy as np
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

IGNORE = -100

## 1. Masked language modeling

`mlm_mask(ids, mask_prob, mask_id, vocab_size, generator)` theo công thức của BERT, với `ids` là `LongTensor` bất kỳ shape:

1. Chọn mỗi vị trí độc lập với xác suất `mask_prob` (dùng `torch.rand(ids.shape, generator=generator) < mask_prob`).
2. `labels` bằng `ids` ở vị trí được chọn, `IGNORE` (-100) ở mọi vị trí khác.
3. Trong các vị trí được chọn: 80% thay bằng `mask_id`, 10% thay bằng một token ngẫu nhiên trong `[0, vocab_size)`, 10%
   giữ nguyên. Vị trí không được chọn giữ nguyên. Mọi số ngẫu nhiên lấy từ `generator`.

Trả `(inputs, labels)`, không sửa `ids` tại chỗ.

In [ ]:
def mlm_mask(ids: torch.Tensor, mask_prob: float, mask_id: int, vocab_size: int, generator: torch.Generator):
    selected = torch.rand(ids.shape, generator=generator) < mask_prob
    labels = torch.where(selected, ids, torch.full_like(ids, IGNORE))
    kind = torch.rand(ids.shape, generator=generator)
    random_tokens = torch.randint(vocab_size, ids.shape, generator=generator)
    inputs = ids.clone()
    inputs[selected & (kind < 0.8)] = mask_id
    swap = selected & (kind >= 0.8) & (kind < 0.9)
    inputs[swap] = random_tokens[swap]
    return inputs, labels

## 2. Một khối Transformer, hai cách dùng

- `attention_mask(T, causal) -> BoolTensor (T, T)`: `True` ở cặp (query $$i$$, key $$j$$) **bị cấm** (quy ước `attn_mask` kiểu bool
  của `nn.TransformerEncoderLayer`). Nhân quả: cấm $$j > i$$; hai chiều: không cấm gì.
- `TinyTransformer(vocab_size, block_size, dim, n_layer, n_head, causal)`: `tok_emb` + `pos_emb` (học được) → `n_layer` lớp
  `nn.TransformerEncoderLayer(dim, n_head, 4 * dim, dropout=0.0, activation="gelu", batch_first=True, norm_first=True)`
  gói trong `nn.TransformerEncoder` (thuộc tính `encoder`, `enable_nested_tensor=False`) → `ln_f` (`nn.LayerNorm`) →
  `head` (`nn.Linear` không bias, **dùng chung** trọng số với `tok_emb`). `forward(idx)` trả logits `(B, T, vocab_size)`,
  truyền `attention_mask(T, self.causal)` vào encoder. Attention bạn đã tự viết ở bài Basic 01; ở đây dùng lớp của PyTorch
  để thấy rằng BERT và GPT chỉ khác nhau ở một tham số.
- `lm_loss(logits, ids)`: GPT, cross-entropy của logits vị trí $$t$$ với token $$t + 1$$ (bỏ vị trí cuối).
- `mlm_loss(logits, labels)`: BERT, cross-entropy chỉ trên vị trí có nhãn (`ignore_index=IGNORE`).
- `masked_accuracy(logits, labels)`: tỉ lệ argmax đúng trên các vị trí `labels != IGNORE`, kiểu `float`.

In [ ]:
def attention_mask(T: int, causal: bool) -> torch.Tensor:
    if causal:
        return torch.ones(T, T, dtype=torch.bool).triu(1)
    return torch.zeros(T, T, dtype=torch.bool)


class TinyTransformer(nn.Module):
    def __init__(self, vocab_size: int, block_size: int, dim: int, n_layer: int, n_head: int, causal: bool):
        super().__init__()
        self.causal = causal
        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Embedding(block_size, dim)
        layer = nn.TransformerEncoderLayer(dim, n_head, 4 * dim, dropout=0.0, activation="gelu", batch_first=True,
                                           norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layer, enable_nested_tensor=False)
        self.ln_f = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        nn.init.normal_(self.tok_emb.weight, std=0.02)
        nn.init.normal_(self.pos_emb.weight, std=0.02)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        T = idx.shape[1]
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        x = self.encoder(x, mask=attention_mask(T, self.causal).to(idx.device))
        return self.head(self.ln_f(x))


def lm_loss(logits: torch.Tensor, ids: torch.Tensor) -> torch.Tensor:
    return F.cross_entropy(logits[:, :-1].reshape(-1, logits.shape[-1]), ids[:, 1:].reshape(-1))


def mlm_loss(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    return F.cross_entropy(logits.reshape(-1, logits.shape[-1]), labels.reshape(-1), ignore_index=IGNORE)


def masked_accuracy(logits: torch.Tensor, labels: torch.Tensor) -> float:
    keep = labels != IGNORE
    return (logits.argmax(-1)[keep] == labels[keep]).float().mean().item()

## 3. Thí nghiệm

Dữ liệu và tokenizer giống bài Intermediate 01 (1 000 bài train, 100 bài validation của module 04, BPE 8 000), thêm một
token `[MASK]` mang id 8 000. Hai model cùng kiến trúc, cùng số tham số, cùng số bước và batch; chỉ khác `causal` và loss.

In [ ]:
# explore: dữ liệu, tokenizer, token [MASK]
def long_articles(path, min_chars):
    texts = pq.read_table(path, columns=["text"]).column("text").to_pylist()
    return [unicodedata.normalize("NFC", t) for t in texts if len(t) >= min_chars]


articles = long_articles(SHARD, 2000)
perm = np.random.RandomState(0).permutation(len(articles))
train_text = "\n\n".join(articles[i] for i in perm[:1000])
val_text = "\n\n".join(articles[i] for i in perm[1000:1100])
tok = Tokenizer(models.BPE())
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder = decoders.ByteLevel()
tok.train_from_iterator([train_text], trainers.BpeTrainer(vocab_size=8000, initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))
MASK_ID, VOCAB = 8000, 8001
train_ids = torch.tensor(tok.encode(train_text).ids)
val_ids = torch.tensor(tok.encode(val_text).ids)
print(f"train {len(train_ids):,} token, val {len(val_ids):,} token")

In [ ]:
# explore: train GPT (causal, next token) và BERT (hai chiều, MLM 15%) cùng ngân sách
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(4)
DIM, N_LAYER, STEPS = (256, 4, 3000) if DEVICE == "cuda" else (128, 4, 800)
BLOCK, BATCH = 128, 32


def batch(data, g):
    starts = torch.randint(len(data) - BLOCK, (BATCH,), generator=g)
    return torch.stack([data[s:s + BLOCK] for s in starts])


def train(causal: bool):
    torch.manual_seed(0)
    model = TinyTransformer(VOCAB, BLOCK, DIM, N_LAYER, 4, causal).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, betas=(0.9, 0.95), weight_decay=0.1)
    g, mg = torch.Generator().manual_seed(0), torch.Generator().manual_seed(1)
    start, losses = time.perf_counter(), []
    for step in range(STEPS):
        for group in opt.param_groups:
            group["lr"] = 1e-3 * min(1.0, (step + 1) / 100) * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * step / STEPS)))
        ids = batch(train_ids, g)
        if causal:
            loss = lm_loss(model(ids.to(DEVICE)), ids.to(DEVICE))
        else:
            inputs, labels = mlm_mask(ids, 0.15, MASK_ID, 8000, mg)
            loss = mlm_loss(model(inputs.to(DEVICE)), labels.to(DEVICE))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        losses.append(loss.item())
    name = "GPT " if causal else "BERT"
    print(f"{name}: {sum(p.numel() for p in model.parameters()):,} tham số, loss cuối {np.mean(losses[-50:]):.3f},"
          f" {time.perf_counter() - start:.0f} s")
    return model.eval(), losses


gpt, gpt_losses = train(causal=True)
bert, bert_losses = train(causal=False)

In [ ]:
# explore: điền chỗ trống trên validation, hai cách: (1) che 15% cùng lúc như lúc train BERT, (2) mỗi lần che đúng một token
g = torch.Generator().manual_seed(2)
n_win = (len(val_ids) - 1) // BLOCK
windows = val_ids[:n_win * BLOCK].view(n_win, BLOCK)


@torch.no_grad()
def gpt_predictions(w):
    shifted = gpt(w.to(DEVICE)).float().cpu()
    return torch.cat([torch.zeros_like(shifted[:, :1]), shifted[:, :-1]], dim=1)


hidden = torch.rand(windows.shape, generator=g) < 0.15
hidden[:, 0] = False
labels = torch.where(hidden, windows, torch.full_like(windows, IGNORE))
bert_logits, gpt_logits = [], []
with torch.no_grad():
    for i in range(0, n_win, 64):
        w = windows[i:i + 64]
        bert_logits.append(bert(torch.where(hidden[i:i + 64], MASK_ID, w).to(DEVICE)).float().cpu())
        gpt_logits.append(gpt_predictions(w))
acc_many = (masked_accuracy(torch.cat(bert_logits), labels), masked_accuracy(torch.cat(gpt_logits), labels))

PER_WINDOW = 8
positions = torch.randint(1, BLOCK, (n_win, PER_WINDOW), generator=g)
gpt_all = torch.cat([gpt_predictions(windows[i:i + 64]) for i in range(0, n_win, 64)])
bert_right = gpt_right = 0
with torch.no_grad():
    for j in range(PER_WINDOW):
        pos = positions[:, j]
        rows = torch.arange(n_win)
        target = windows[rows, pos]
        for i in range(0, n_win, 64):
            w = windows[i:i + 64].clone()
            w[torch.arange(len(w)), pos[i:i + 64]] = MASK_ID
            pred = bert(w.to(DEVICE)).float().cpu()[torch.arange(len(w)), pos[i:i + 64]].argmax(-1)
            bert_right += int((pred == target[i:i + 64]).sum())
        gpt_right += int((gpt_all[rows, pos].argmax(-1) == target).sum())
n_single = n_win * PER_WINDOW
acc_single = (bert_right / n_single, gpt_right / n_single)
print(f"(1) che 15% cùng lúc ({int(hidden.sum()):,} vị trí): BERT {acc_many[0]:.3f} | GPT {acc_many[1]:.3f}")
print(f"(2) che một token mỗi lần ({n_single:,} vị trí):    BERT {acc_single[0]:.3f} | GPT {acc_single[1]:.3f}")

In [ ]:
# explore: ví dụ điền một từ, và thử bắt cả hai model sinh văn bản
sentence = "Hà Nội là thủ đô của nước Việt Nam."
ids = torch.tensor([tok.encode(sentence).ids])
pos = next(i for i, t in enumerate(ids[0].tolist()) if tok.decode([t]).strip() == "đô")
masked = ids.clone()
masked[0, pos] = MASK_ID
with torch.no_grad():
    top_bert = bert(masked.to(DEVICE))[0, pos].topk(5).indices.tolist()
    top_gpt = gpt(ids[:, :pos].to(DEVICE))[0, -1].topk(5).indices.tolist()
print("câu:", sentence.replace("đô", "[MASK]", 1))
print("BERT đoán:", [tok.decode([t]) for t in top_bert])
print("GPT đoán: ", [tok.decode([t]) for t in top_gpt])

prompt = tok.encode("Hà Nội là").ids
for name, model in [("GPT", gpt), ("BERT", bert)]:
    seq = list(prompt)
    with torch.no_grad():
        for _ in range(25):
            if name == "GPT":
                logits = model(torch.tensor([seq[-BLOCK:]], device=DEVICE))[0, -1]
            else:
                logits = model(torch.tensor([(seq + [MASK_ID])[-BLOCK:]], device=DEVICE))[0, -1]
            logits[MASK_ID:] = -float("inf")
            seq.append(int(logits.argmax()))
    print(f"{name} sinh greedy: {tok.decode(seq)}")

In [ ]:
# explore: đường loss của hai model (cùng đơn vị nat/token nhưng khác nhiệm vụ: không so trực tiếp được)
import matplotlib.pyplot as plt

smooth = lambda v: np.convolve(v, np.ones(25) / 25, mode="valid")
plt.figure(figsize=(6, 3.5))
plt.plot(smooth(gpt_losses), label="GPT: next token")
plt.plot(smooth(bert_losses), label="BERT: MLM 15%")
plt.xlabel("bước"); plt.ylabel("loss train"); plt.legend(); plt.tight_layout(); plt.show()

## 4. Đọc kết quả

So với mục 3 phần lý thuyết.

- **Điền chỗ trống, cách 1** (che 15% cùng lúc): ngữ cảnh của BERT bị đục lỗ ở cả hai phía, còn GPT thấy đúng mọi token bên
  trái. Model nào thắng trong lần chạy của bạn, và điều đó có khớp với giả thuyết "nhìn hai phía thì điền giỏi hơn" không?
- **Cách 2** (mỗi lần một chỗ trống, mọi token khác thật): ngữ cảnh hai phía là thông tin thật ("thủ [MASK] của nước" gần như
  chỉ có một đáp án). So chênh lệch giữa hai model với cách 1. Nhớ rằng ở cùng số bước, mỗi bước BERT chỉ có loss trên khoảng
  15% token, GPT có loss trên mọi token, gần 7 lần tín hiệu: **hiệu quả dữ liệu** là một biến của thí nghiệm. BERT gốc train 1
  triệu bước.
- **Sinh văn bản**: BERT chưa bao giờ được train để đoán token **cuối** khi bên phải không có gì. Đọc văn bản hai model sinh ra
  và giải thích vì sao mọi LLM sinh văn bản đều là decoder.
- Loss train của hai bên khác nhiệm vụ nên không so trực tiếp; điều so được là chất lượng trên cùng một việc.

Sau khi chạy hết notebook, làm phần **trắc nghiệm** của bài trong app.